# 🚲 AdventureWorks Sales Analysis

**ETL Pipeline: من MySQL إلى Star Schema**

---

## 📋 نظرة عامة

مشروع كامل يحوّل بيانات مبيعات AdventureWorks من قاعدة بيانات MySQL إلى Star Schema جاهز للتحليل والتصور، مع فحص جودة شامل قبل وبعد الدمج.

**الناتج**: 121,317 صف | 22 عمودًا | 4 جداول Star Schema

---

## 🗂️ فهرس المحتويات

| # | القسم |
|---|---|
| 1 | استيراد المكتبات |
| 2 | الاتصال بقاعدة البيانات |
| 3 | استكشاف الجداول |
| 4 | فحص جودة المصدر |
| 5 | جلب البيانات (Extract) |
| 6 | التحويل (Transform) |
| 7 | بناء Star Schema |
| 8 | التصدير (Load) |

---

## 🛠️ التقنيات

- **Python 3.13** + pandas + SQLAlchemy
- **MariaDB** (AdventureWorks 2019)
- **Jupyter Notebook** للـETL

---

**: بسام طارق**

In [2]:
# ============================================================
# الخلية 1: استيراد المكتبات
# ============================================================
import os
import pymysql
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

print("✅ تم استيراد المكتبات")

✅ تم استيراد المكتبات


In [3]:
# ============================================================
# الخلية 2: الاتصال بقاعدة البيانات (آمن عبر .env)
# ============================================================

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --- تحميل الإعدادات من .env ---
load_dotenv()

# --- قراءة الإعدادات (بلا قيم افتراضية) ---
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

# --- التحقق من وجودها ---
if not all([DB_USER, DB_HOST, DB_NAME]):
    raise ValueError(
        "❌ لم يتم العثور على إعدادات الاتصال.\n"
        "تأكد من وجود ملف .env في مجلد المشروع.\n"
        "راجع .env.example كنموذج."
    )

# --- إنشاء محرك الاتصال ---
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}"
)

# --- اختبار الاتصال ---
with engine.connect() as conn:
    version = conn.execute(text("SELECT VERSION();")).fetchone()[0]

print("✅ تم الاتصال بقاعدة البيانات")
print(f"📌 الإصدار: {version}")

✅ تم الاتصال بقاعدة البيانات
📌 الإصدار: 10.4.32-MariaDB


In [9]:
# ============================================================
# الخلية 3: عرض جميع الجداول
# ============================================================
tables = pd.read_sql("SHOW TABLES;", engine)
print(f"📊 عدد الجداول: {len(tables)}\n")
print(tables.to_string(index=False))

📊 عدد الجداول: 71

                             Tables_in_adventure
                              dbo_awbuildversion
                                 dbo_databaselog
                                    dbo_errorlog
                       humanresources_department
                         humanresources_employee
        humanresources_employeedepartmenthistory
               humanresources_employeepayhistory
                     humanresources_jobcandidate
                            humanresources_shift
                                  person_address
                              person_addresstype
                           person_businessentity
                    person_businessentityaddress
                    person_businessentitycontact
                              person_contacttype
                            person_countryregion
                             person_emailaddress
                                 person_password
                                   person_person
 

In [10]:
# ============================================================
# الخلية 4: اختيار الجداول الأساسية (Pre-flight Check)
# ============================================================

selected_tables = [
    'sales_salesorderheader',
    'sales_salesorderdetail',
    'sales_customer',
    'production_product',
    'production_productsubcategory',
    'production_productcategory'
]

all_tables = set(pd.read_sql("SHOW TABLES;", engine).iloc[:, 0])

found = 0
print("🔍 التحقق من وجود الجداول:\n")
for t in selected_tables:
    exists = t in all_tables
    print(f"   {'✅' if exists else '❌'} {t}")
    found += exists

print(f"\n📊 {found}/{len(selected_tables)} جدول موجود")

🔍 التحقق من وجود الجداول:

   ✅ sales_salesorderheader
   ✅ sales_salesorderdetail
   ✅ sales_customer
   ✅ production_product
   ✅ production_productsubcategory
   ✅ production_productcategory

📊 6/6 جدول موجود


In [12]:
# ============================================================
# الخلية 5: بناء schema_df (بنية كل الجداول)
# ============================================================

schema_list = []
for table in selected_tables:
    cols = pd.read_sql(f"DESCRIBE {table};", engine)
    cols.insert(0, "Table", table)
    schema_list.append(cols)

schema_df = pd.concat(schema_list, ignore_index=True)

# --- عرض المفاتيح + الأعمدة التي تقبل NULL ---
rows = []
for table in selected_tables:
    g = schema_df[schema_df["Table"] == table]
    pk = ", ".join(g[g["Key"] == "PRI"]["Field"].tolist()) or "—"
    nulls = ", ".join(g[g["Null"] == "YES"]["Field"].tolist()) or "—"
    rows.append({"الجدول": table, "المفاتيح الأساسية": pk, "أعمدة تقبل NULL": nulls})

schema_summary = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
display(schema_summary)

,الجدول,المفاتيح الأساسية,أعمدة تقبل NULL
0,sales_salesorderheader,SalesOrderID,"ShipDate, PurchaseOrderNumber, AccountNumber, SalesPersonID, TerritoryID, CreditCardID, CreditCardApprovalCode, CurrencyRateID, Comment"
1,sales_salesorderdetail,"SalesOrderID, SalesOrderDetailID",CarrierTrackingNumber
2,sales_customer,CustomerID,"PersonID, StoreID, TerritoryID"
3,production_product,ProductID,"Color, Size, SizeUnitMeasureCode, WeightUnitMeasureCode, Weight, ProductLine, Class, Style, ProductSubcategoryID, ProductModelID, SellEndDate, DiscontinuedDate"
4,production_productsubcategory,ProductSubcategoryID,—
5,production_productcategory,ProductCategoryID,—


In [13]:
# ============================================================
# الخلية 6: استخراج العلاقات بين جداولنا
# ============================================================

# --- كل العلاقات في القاعدة ---
fks = pd.read_sql("""
    SELECT TABLE_NAME AS from_table, COLUMN_NAME AS from_column,
           REFERENCED_TABLE_NAME AS to_table, REFERENCED_COLUMN_NAME AS to_column
    FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
    WHERE TABLE_SCHEMA = 'adventure'
      AND REFERENCED_TABLE_NAME IS NOT NULL
""", engine)

# --- العلاقات بين جداولنا فقط ---
relevant_fks = fks[
    (fks['from_table'].isin(selected_tables)) &
    (fks['to_table'].isin(selected_tables))
].reset_index(drop=True)

# --- إضافة العلاقة المنطقية غير المباشرة ---
missing_fk = pd.DataFrame([{
    "from_table": "sales_salesorderdetail",
    "from_column": "ProductID",
    "to_table": "production_product",
    "to_column": "ProductID"
}])

all_fks = pd.concat([relevant_fks, missing_fk], ignore_index=True)

print(f"🔗 العلاقات بين جداولنا: {len(all_fks)}\n")
print(all_fks.to_string(index=False))

🔗 العلاقات بين جداولنا: 5

                   from_table          from_column                      to_table            to_column
           production_product ProductSubcategoryID production_productsubcategory ProductSubcategoryID
production_productsubcategory    ProductCategoryID    production_productcategory    ProductCategoryID
       sales_salesorderdetail         SalesOrderID        sales_salesorderheader         SalesOrderID
       sales_salesorderheader           CustomerID                sales_customer           CustomerID
       sales_salesorderdetail            ProductID            production_product            ProductID


In [14]:
# ============================================================
# الخلية 7: فحص سلامة المراجع (Referential Integrity)
# ============================================================

print("🔍 فحص سلامة المراجع\n")
orphans_total = 0

for _, fk in all_fks.iterrows():
    q = f"""
        SELECT COUNT(*) FROM {fk['from_table']} s
        LEFT JOIN {fk['to_table']} r 
            ON s.{fk['from_column']} = r.{fk['to_column']}
        WHERE s.{fk['from_column']} IS NOT NULL 
          AND r.{fk['to_column']} IS NULL
    """
    n = pd.read_sql(q, engine).iloc[0, 0]
    orphans_total += n
    flag = "✅" if n == 0 else "⚠️"
    print(f"   {flag} {fk['from_table']}.{fk['from_column']} → {fk['to_table']}: {n}")

print(f"\n{'✅ كل العلاقات سليمة' if orphans_total == 0 else f'⚠️ {orphans_total} NULL'}")

🔍 فحص سلامة المراجع

   ✅ production_product.ProductSubcategoryID → production_productsubcategory: 0
   ✅ production_productsubcategory.ProductCategoryID → production_productcategory: 0
   ✅ sales_salesorderdetail.SalesOrderID → sales_salesorderheader: 0
   ✅ sales_salesorderheader.CustomerID → sales_customer: 0
   ✅ sales_salesorderdetail.ProductID → production_product: 0

✅ كل العلاقات سليمة


---
## 5️⃣ جلب البيانات (Extract)

استعلام SQL واحد يدمج **6 جداول**:

| الجدول | الدور |
|---|---|
| `sales_salesorderheader` | رؤوس الطلبات |
| `sales_salesorderdetail` | تفاصيل الطلبات |
| `sales_customer` | العملاء |
| `production_product` | المنتجات |
| `production_productsubcategory` | التصنيف الفرعي |
| `production_productcategory` | التصنيف الرئيسي |

**الفلتر**: كل البيانات (2011-2014).

In [15]:
# ============================================================
# الخلية 8: بناء الاستعلام النهائي (Extract)
# ============================================================

query = """
SELECT 
    soh.SalesOrderID, soh.OrderDate, soh.CustomerID,
    soh.SalesPersonID, soh.TerritoryID, soh.OnlineOrderFlag,
    sod.SalesOrderDetailID, sod.ProductID, sod.OrderQty,
    sod.UnitPrice, sod.UnitPriceDiscount, sod.LineTotal,
    p.Name  AS ProductName, p.ListPrice, p.StandardCost,
    ps.Name AS SubCategoryName,
    pc.Name AS CategoryName
FROM sales_salesorderheader soh
INNER JOIN sales_salesorderdetail sod ON soh.SalesOrderID = sod.SalesOrderID
INNER JOIN sales_customer c           ON soh.CustomerID = c.CustomerID
INNER JOIN production_product p       ON sod.ProductID = p.ProductID
LEFT JOIN production_productsubcategory ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
LEFT JOIN production_productcategory pc    ON ps.ProductCategoryID = pc.ProductCategoryID
"""

print("✅ تم بناء الاستعلام النهائي")

✅ تم بناء الاستعلام النهائي


In [16]:
# ============================================================
# الخلية 9: جلب البيانات
# ============================================================

print("🔄 جاري جلب البيانات...")
df = pd.read_sql(query, engine)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

print(f"✅ تم جلب {len(df):,} صف")
print(f"📊 عدد الأعمدة: {len(df.columns)}")

🔄 جاري جلب البيانات...
✅ تم جلب 121,317 صف
📊 عدد الأعمدة: 17


In [17]:
# ============================================================
# الخلية 10: فحص جودة الناتج (بعد الدمج)
# ============================================================

print("🔍 فحص جودة الناتج\n")

# الحجم
print(f"📊 الصفوف: {len(df):,} | الأعمدة: {len(df.columns)}")

# الحبيبة
dupes = df.duplicated(["SalesOrderID", "SalesOrderDetailID"]).sum()
print(f"\n🔑 تكرار (OrderID, DetailID): {dupes}")

# NULLs الحرجة
critical = ["SalesOrderID", "OrderDate", "CustomerID", "ProductID",
            "OrderQty", "UnitPrice", "LineTotal", "ProductName"]
print(f"\n🔍 NULLs في الأعمدة الحرجة:")
for col in critical:
    n = df[col].isna().sum()
    print(f"   {'✅' if n == 0 else '⚠️'} {col:<20} {n}")

# منطقية القيم
print(f"\n📈 منطقية القيم:")
print(f"   {'✅' if (df['OrderQty'] > 0).all() else '⚠️'} OrderQty > 0")
print(f"   {'✅' if (df['LineTotal'] >= 0).all() else '⚠️'} LineTotal >= 0")

🔍 فحص جودة الناتج

📊 الصفوف: 121,317 | الأعمدة: 17

🔑 تكرار (OrderID, DetailID): 0

🔍 NULLs في الأعمدة الحرجة:
   ✅ SalesOrderID         0
   ✅ OrderDate            0
   ✅ CustomerID           0
   ✅ ProductID            0
   ✅ OrderQty             0
   ✅ UnitPrice            0
   ✅ LineTotal            0
   ✅ ProductName          0

📈 منطقية القيم:
   ✅ OrderQty > 0
   ✅ LineTotal >= 0


In [18]:
# ============================================================
# الخلية 11: الملخص النهائي
# ============================================================

print("=" * 55)
print("📊 ملخص البيانات")
print("=" * 55)
print(f"   عدد الصفوف:      {len(df):,}")
print(f"   عدد الأعمدة:     {len(df.columns)}")
print(f"   النطاق الزمني:   {df['OrderDate'].min().date()} → {df['OrderDate'].max().date()}")
print(f"   عدد العملاء:     {df['CustomerID'].nunique():,}")
print(f"   عدد المنتجات:    {df['ProductID'].nunique():,}")
print(f"   إجمالي الإيرادات: {df['LineTotal'].sum():,.2f}")
print("=" * 55)

📊 ملخص البيانات
   عدد الصفوف:      121,317
   عدد الأعمدة:     17
   النطاق الزمني:   2011-05-30 → 2014-06-29
   عدد العملاء:     19,119
   عدد المنتجات:    266
   إجمالي الإيرادات: 109,846,381.40


In [19]:
# ============================================================
# الخلية 12: إضافة الأعمدة التحليلية
# ============================================================

# --- حساب الربح ---
df['Profit'] = df['LineTotal'] - (df['OrderQty'] * df['StandardCost'])

# --- تصنيف الربحية ---
df['IsProfitable'] = df['Profit'] > 0

# --- أعمدة زمنية ---
df['OrderYear']  = df['OrderDate'].dt.year
df['OrderMonth'] = df['OrderDate'].dt.month

# --- التحقق ---
print("✅ تم إضافة الأعمدة: Profit, IsProfitable, OrderYear, OrderMonth\n")
print(f"💰 إجمالي الربح: {df['Profit'].sum():,.2f}")
print(f"📊 هامش الربح:   {(df['Profit'].sum() / df['LineTotal'].sum() * 100):.2f}%")
print(f"✅ صفوف رابحة:    {df['IsProfitable'].sum():,}")
print(f"⚠️ صفوف ترويجية: {(~df['IsProfitable']).sum():,}")
print(f"\n📅 توزيع السنوات:")
print(df['OrderYear'].value_counts().sort_index().to_string())

✅ تم إضافة الأعمدة: Profit, IsProfitable, OrderYear, OrderMonth

💰 إجمالي الربح: 9,371,903.63
📊 هامش الربح:   8.53%
✅ صفوف رابحة:    92,156
⚠️ صفوف ترويجية: 29,161

📅 توزيع السنوات:
OrderYear
2011     6588
2012    20827
2013    56711
2014    37191


In [20]:
# ============================================================
# الخلية 13: تصدير الملف  للتحاليل 
# ============================================================

from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

df.to_csv(OUTPUT / "adventureworks.csv", index=False)
df.to_parquet(OUTPUT / "adventureworks.parquet", index=False)

print(f"✅ تم التصدير:")
print(f"   📄 adventureworks.csv      ({len(df):,} صف، {len(df.columns)} عمود)")
print(f"   📄 adventureworks.parquet")

✅ تم التصدير:
   📄 adventureworks.csv      (121,317 صف، 21 عمود)
   📄 adventureworks.parquet


In [13]:
# ============================================================
# الخلية 14: بناء Star Schema (4 جداول)
# ============================================================
# --- 1) جدول الحقائق ---
fact_sales = df[[
    'SalesOrderID', 'SalesOrderDetailID', 'OrderDate',
    'CustomerID', 'ProductID', 'SalesPersonID', 'TerritoryID',
    'OrderQty', 'UnitPrice', 'UnitPriceDiscount', 'LineTotal',
    'Profit', 'OnlineOrderFlag'
]].copy()

# --- 2) بُعد المنتج ---
dim_product = (df[['ProductID', 'ProductName', 'SubCategoryName',
                   'CategoryName', 'ListPrice', 'StandardCost']]
               .drop_duplicates(subset=['ProductID'])
               .reset_index(drop=True))

# --- 3) بُعد العميل ---
dim_customer = (df[['CustomerID', 'TerritoryID']]
                .drop_duplicates(subset=['CustomerID'])
                .reset_index(drop=True))

# --- 4) بُعد التاريخ ---
dim_date = pd.DataFrame({
    'Date': pd.date_range(df['OrderDate'].min(), df['OrderDate'].max(), freq='D')
})
dim_date['Year']        = dim_date['Date'].dt.year
dim_date['Quarter']     = dim_date['Date'].dt.quarter
dim_date['MonthNumber'] = dim_date['Date'].dt.month
dim_date['MonthName']   = dim_date['Date'].dt.strftime('%B')
dim_date['MonthYear']   = dim_date['Date'].dt.strftime('%Y-%m')
dim_date['DayName']     = dim_date['Date'].dt.day_name()
dim_date['IsWeekend']   = dim_date['Date'].dt.dayofweek >= 5

# --- التحقق ---
print("✅ تم بناء Star Schema:\n")
for name, table in [('fact_sales', fact_sales), ('dim_product', dim_product),
                    ('dim_customer', dim_customer), ('dim_date', dim_date)]:
    print(f"   📄 {name:<15} {len(table):>7,} صف، {len(table.columns)} عمود")

✅ تم بناء Star Schema:

   📄 fact_sales      121,317 صف، 13 عمود
   📄 dim_product         266 صف، 6 عمود
   📄 dim_customer     19,119 صف، 2 عمود
   📄 dim_date          1,127 صف، 8 عمود


---
## 8️⃣ التصدير (Load)

نصدّر البيانات بصيغتين:

- **CSV** — للتحميل في Streamlit وPower BI.
- **Parquet** — للأرشيف والتحليل السريع.

**ملاحظة تقنية**: راجع `TECHNICAL_NOTES.md` لمشكلة `pyarrow` على Windows.

In [14]:
# ============================================================
# الخلية 15: التصدير لـ Power BI
# ============================================================

from pathlib import Path

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

# --- تصدير 4 ملفات CSV + Parquet ---
exports = {
    'fact_sales':   fact_sales,
    'dim_product':  dim_product,
    'dim_customer': dim_customer,
    'dim_date':     dim_date,
}

for name, table in exports.items():
    table.to_csv(OUTPUT / f"powerbi_{name}.csv", index=False)
    table.to_parquet(OUTPUT / f"powerbi_{name}.parquet", index=False)
    print(f"   📄 {name}.csv + .parquet")

print(f"\n✅ تم التصدير إلى: {OUTPUT.resolve()}")

   📄 fact_sales.csv + .parquet
   📄 dim_product.csv + .parquet
   📄 dim_customer.csv + .parquet
   📄 dim_date.csv + .parquet

✅ تم التصدير إلى: D:\Data_properties\output


---
## 9️⃣ المقاييس (Measures)

نبني المقاييس كدوال Python — مكافئ `DAX Measures` في Power BI.

**المفهوم الأساسي**: نفس الدالة + فلتر مختلف = سؤال مختلف (`Filter Context`).

| المقياس | مكافئ DAX |
|---|---|
| `total_sales()` | `SUM(LineTotal)` |
| `profit_margin()` | `DIVIDE([Profit], [Sales])` |
| `order_count()` | `DISTINCTCOUNT(SalesOrderID)` |

In [21]:
# ============================================================
# الخلية 16: المقاييس (مكافئ DAX Measures)
# ============================================================

def safe_divide(a, b):
    """قسمة آمنة — مكافئ DIVIDE() في DAX"""
    return a / b if b else 0


# --- المقاييس الأساسية ---
def total_sales(data):        return data['LineTotal'].sum()
def total_profit(data):       return data['Profit'].sum()
def total_qty(data):          return data['OrderQty'].sum()
def order_count(data):        return data['SalesOrderID'].nunique()
def unique_customers(data):   return data['CustomerID'].nunique()

# --- المقاييس النسبية ---
def profit_margin(data):
    return safe_divide(total_profit(data), total_sales(data)) * 100

def avg_order_value(data):
    return safe_divide(total_sales(data), order_count(data))

def avg_qty_per_order(data):
    return safe_divide(total_qty(data), order_count(data))


# --- عرض KPIs ---
def show_kpis(data, title="KPIs"):
    print(f"\n{'=' * 50}")
    print(f"📈 {title}")
    print('=' * 50)
    print(f"   💰 إجمالي المبيعات:     {total_sales(data):>15,.0f}")
    print(f"   💵 إجمالي الربح:        {total_profit(data):>15,.0f}")
    print(f"   📊 هامش الربح:          {profit_margin(data):>14.2f}%")
    print(f"   🧾 عدد الطلبات:         {order_count(data):>15,}")
    print(f"   👥 عدد العملاء:         {unique_customers(data):>15,}")
    print(f"   📦 إجمالي الكمية:       {total_qty(data):>15,}")
    print(f"   🛒 متوسط قيمة الطلب:    {avg_order_value(data):>15,.2f}")
    print(f"   📦 متوسط الكمية/طلب:    {avg_qty_per_order(data):>15,.2f}")
    print('=' * 50)


# --- التشغيل على كل البيانات ---
show_kpis(df, "كل البيانات")

# --- إظهار Filter Context ---
show_kpis(df[df['CategoryName'] == 'Bikes'], "الدراجات فقط")
show_kpis(df[df['OrderYear'] == 2013], "سنة 2013 فقط")


📈 كل البيانات
   💰 إجمالي المبيعات:         109,846,381
   💵 إجمالي الربح:              9,371,904
   📊 هامش الربح:                    8.53%
   🧾 عدد الطلبات:                  31,465
   👥 عدد العملاء:                  19,119
   📦 إجمالي الكمية:               274,914
   🛒 متوسط قيمة الطلب:           3,491.07
   📦 متوسط الكمية/طلب:               8.74

📈 الدراجات فقط
   💰 إجمالي المبيعات:          94,651,173
   💵 إجمالي الربح:              7,936,394
   📊 هامش الربح:                    8.38%
   🧾 عدد الطلبات:                  18,368
   👥 عدد العملاء:                   9,719
   📦 إجمالي الكمية:                90,268
   🛒 متوسط قيمة الطلب:           5,153.05
   📦 متوسط الكمية/طلب:               4.91

📈 سنة 2013 فقط
   💰 إجمالي المبيعات:          43,652,266
   💵 إجمالي الربح:              3,367,898
   📊 هامش الربح:                    7.72%
   🧾 عدد الطلبات:                  14,234
   👥 عدد العملاء:                  11,141
   📦 إجمالي الكمية:               131,926
   🛒 متوسط قيمة الطلب:       